In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CensusStreaming")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark

In [2]:
from pyspark.sql.types import *

schema_defined = StructType([
    StructField('age', LongType(), True),
    StructField('workclass', StringType(), True),
    StructField('fnlwgt', LongType(), True),
    StructField('education', StringType(), True),
    StructField('education-num', LongType(), True),
    StructField('marital-status', StringType(), True),
    StructField('occupation', StringType(), True),
    StructField('relationship', StringType(), True),
    StructField('race', StringType(), True),
    StructField('sex', StringType(), True),
    StructField('capital-gain', LongType(), True),
    StructField('capital-loss', LongType(), True),
    StructField('hours-per-week', LongType(), True),
    StructField('native-country', StringType(), True),
    StructField('income', StringType(), True)
])

In [3]:
STREAM_PATH = "/opt/spark/stream-read"
read_stream = (spark.readStream
                         .format("csv")
                         .schema(schema_defined) 
                         .option("header", "true")
                         .option("maxFilesPerTrigger", 1)
                         .load(STREAM_PATH))
stream_memory_query=(read_stream.writeStream
              .outputMode("append")
              .format("memory")
              .queryName("stream_data_check")
              .trigger(processingTime="5 seconds")
              .start())
                         
print("Streaming query started, writing to memory table 'stream_data_check'.")
print("Waiting for data to be processed and appear in the table...")

Streaming query started, writing to memory table 'stream_data_check'.
Waiting for data to be processed and appear in the table...


In [5]:
import time
from IPython.display import display

# Give the stream some time to process the initial files and the new file
print("Fetching data from 'stream_data' table every 5")
for i in range(30):
    if stream_memory_query.isActive:
        # Query the memory table to see what the stream has processed
        print(f"--- Snapshot {i+1} at {time.strftime('%H:%M:%S')} ---")
        display(spark.sql("SELECT count(*) FROM stream_data_check").show(truncate=False))
        time.sleep(5) # Wait for the next trigger
    else:
        print("Stream query became inactive.")
        break

print("Finished observing memory table.")

# Stop the query after observation
if stream_memory_query.isActive:
    stream_memory_query.stop()
    print("Streaming query 'stream_data_check' explicitly stopped.")

Fetching data from 'stream_data' table every 5
Stream query became inactive.
Finished observing memory table.


In [ ]:
memory_query = (
    read_stream.writeStream
    .format("memory")
    .queryName("stream_table")
    .outputMode("append")
    .start()
)

In [ ]:
spark.sql("SELECT * FROM stream_table LIMIT 10").show()


In [ ]:
from pyspark.sql.functions import count

count_df = read_stream.groupBy().agg(count("*").alias("total_rows"))

In [ ]:
count_query = (
    count_df.writeStream
    .outputMode("complete")
    .format("console")
    .start()
)

In [ ]:
education_df = (
    read_stream
    .groupBy("education")
    .count()
)

In [ ]:
education_query = (
    education_df.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

In [ ]:
from pyspark.sql.functions import count,avg
#Autres agrégations : 
#Personne par sexe
personne_sexe_df =(
    read_stream
    .groupBy("sex")
    .count()

)
personne_sexe_query=(
    personne_sexe_df.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

#personne par pays

personne_pays_df =(
    read_stream
    .groupBy("native-country")
    .count()

)
personne_pays_query=(
    personne_pays_df.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

#Moyenne des heures travaillés

heure_travaille=(
    read_stream
    .groupBy()
    .agg(
        avg("hours-per-week").alias("moyenne_heure")
    )
)

heure_travaille_query=(
    heure_travaille.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

#moyenne âge par profession

moy_age_prof=(
    read_stream
    .groupBy("occupation")
    .agg(
        avg("age").alias("Moyenne âge")
    )
)
moy_age_prof_query=(
    moy_age_prof.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)
